## Импорт данных

In [1]:
import time
import requests
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn.calibration import CalibratedClassifierCV

try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None

In [2]:
def load_binance_klines(
    symbol: str = "BTCUSDT",
    interval: str = "5m",
    limit_total: int = 10_000,
) -> pd.DataFrame:
    """
    Тянем историю с Binance пачками по 1000 свечей, двигаясь назад во времени.
    Возвращает DataFrame с индексом datetime и колонками OHLCV.
    """
    url = "https://api.binance.com/api/v3/klines"
    all_klines = []
    end_time = None
    per_request = 1000

    while len(all_klines) < limit_total:
        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": per_request,
        }

        if end_time is not None:
            params["endTime"] = end_time - 1

        session = requests.Session()
        session.trust_env = False  # не брать proxy из переменных окружения
        
        resp = session.get(url, params=params, timeout=20)
        resp.raise_for_status()
        klines = resp.json()

        if not klines:
            break

        all_klines = klines + all_klines
        end_time = klines[0][0]

        time.sleep(0.1)

    all_klines = all_klines[-limit_total:]

    df_binance = pd.DataFrame(all_klines, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "qav", "num_trades", "taker_base",
        "taker_quote", "ignore"
    ])

    numeric_cols = ["open", "high", "low", "close", "volume"]
    for col in numeric_cols:
        df_binance[col] = df_binance[col].astype(float)

    df_binance["datetime"] = pd.to_datetime(df_binance["open_time"], unit="ms", utc=True)

    df_binance = df_binance[["datetime", "open", "high", "low", "close", "volume"]]
    df_binance = df_binance.sort_values("datetime")
    df_binance = df_binance.drop_duplicates(subset=["datetime"])
    df_binance = df_binance.set_index("datetime")

    return df_binance

## Проверка на пустоты

In [3]:
def validate_ohlcv(df: pd.DataFrame, expected_freq: str = "5min") -> pd.DataFrame:
    """
    Проверяет сортировку, дубли и пропуски по времени.
    """
    x = df.copy()
    x = x.sort_index()
    x = x[~x.index.duplicated(keep="last")]

    full_index = pd.date_range(
        start=x.index.min(),
        end=x.index.max(),
        freq=expected_freq,
        tz=x.index.tz
    )

    missing = full_index.difference(x.index)

    print(f"Количество строк: {len(x)}")
    print(f"Начало: {x.index.min()}")
    print(f"Конец: {x.index.max()}")

    if len(missing) > 0:
        print(f"Пропущено свечей: {len(missing)}")
    else:
        print("Пропусков нет")

    return x

## RSI

In [4]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return rsi


def compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift(1)).abs()
    low_close = (df["low"] - df["close"].shift(1)).abs()

    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = true_range.rolling(period).mean()

    return atr


## Построение признаков

In [5]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()

    # Доходности
    x["ret_1"] = x["close"].pct_change(1)
    x["ret_2"] = x["close"].pct_change(2)
    x["ret_3"] = x["close"].pct_change(3)
    x["ret_5"] = x["close"].pct_change(5)
    x["ret_10"] = x["close"].pct_change(10)
    x["ret_20"] = x["close"].pct_change(20)

    # Свечные признаки
    x["hl_range"] = (x["high"] - x["low"]) / x["close"]
    x["body"] = (x["close"] - x["open"]) / x["open"]
    x["upper_wick"] = (x["high"] - x[["open", "close"]].max(axis=1)) / x["close"]
    x["lower_wick"] = (x[["open", "close"]].min(axis=1) - x["low"]) / x["close"]
    x["close_pos_in_range"] = (
        (x["close"] - x["low"]) / (x["high"] - x["low"]).replace(0, np.nan)
    )

    # EMA
    x["ema_9"] = x["close"].ewm(span=9, adjust=False).mean()
    x["ema_21"] = x["close"].ewm(span=21, adjust=False).mean()
    x["ema_50"] = x["close"].ewm(span=50, adjust=False).mean()

    x["dist_ema_9"] = (x["close"] - x["ema_9"]) / x["ema_9"]
    x["dist_ema_21"] = (x["close"] - x["ema_21"]) / x["ema_21"]
    x["dist_ema_50"] = (x["close"] - x["ema_50"]) / x["ema_50"]

    x["ema_spread_9_21"] = (x["ema_9"] - x["ema_21"]) / x["ema_21"]
    x["ema_spread_21_50"] = (x["ema_21"] - x["ema_50"]) / x["ema_50"]

    # Сила и наклон тренда
    x["ema_slope_9_3"] = x["ema_9"].pct_change(3)
    x["ema_slope_21_5"] = x["ema_21"].pct_change(5)
    x["ema_slope_50_10"] = x["ema_50"].pct_change(10)

    # Momentum
    x["rsi_14"] = compute_rsi(x["close"], period=14)
    x["rsi_delta_3"] = x["rsi_14"].diff(3)

    # ATR / волатильность режима
    x["atr_14"] = compute_atr(x, period=14)
    x["atr_pct_14"] = x["atr_14"] / x["close"]

    x["volatility_10"] = x["ret_1"].rolling(10).std()
    x["volatility_20"] = x["ret_1"].rolling(20).std()
    x["volatility_ratio_10_20"] = x["volatility_10"] / x["volatility_20"].replace(0, np.nan)

    # Объём
    x["volume_ma_20"] = x["volume"].rolling(20).mean()
    x["volume_std_20"] = x["volume"].rolling(20).std()
    x["volume_ratio"] = x["volume"] / x["volume_ma_20"]
    x["volume_zscore_20"] = (
        (x["volume"] - x["volume_ma_20"]) / x["volume_std_20"].replace(0, np.nan)
    )

    # Контекст относительно локальных экстремумов
    x["rolling_high_20"] = x["high"].rolling(20).max()
    x["rolling_low_20"] = x["low"].rolling(20).min()

    x["dist_high_20"] = (x["close"] - x["rolling_high_20"]) / x["rolling_high_20"]
    x["dist_low_20"] = (x["close"] - x["rolling_low_20"]) / x["rolling_low_20"]

    # Дополнительные признаки для analyzer-режима
    x["range_expansion_1"] = (x["high"] - x["low"]) / x["close"]
    x["range_expansion_5"] = x["range_expansion_1"].rolling(5).mean()

    x["breakout_up_20"] = x["close"] / x["rolling_high_20"].replace(0, np.nan) - 1.0
    x["breakout_down_20"] = x["close"] / x["rolling_low_20"].replace(0, np.nan) - 1.0

    x["trend_strength_20"] = (
        x["close"].diff(20).abs()
        / x["close"].diff().abs().rolling(20).sum().replace(0, np.nan)
    )

    # Efficiency ratio (насколько движение направленное, а не шумовое)
    x["efficiency_ratio_10"] = (
        (x["close"] - x["close"].shift(10)).abs()
        / x["close"].diff().abs().rolling(10).sum().replace(0, np.nan)
    )

    # Временной контекст: циклическое кодирование
    hour = x.index.hour
    dayofweek = x.index.dayofweek

    x["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    x["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    x["dow_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    x["dow_cos"] = np.cos(2 * np.pi * dayofweek / 7)

    return x


## Построение таргета

In [6]:
def build_target(
    df: pd.DataFrame,
    horizon: int = 12,
    base_threshold: float = 0.0035,
    atr_mult: float = 0.7,
    atr_col: str = "atr_pct_14",
    min_threshold: float = 0.0030,
    max_threshold: float = 0.0080,
) -> pd.DataFrame:
    """
    0 = DOWN
    1 = UNSURE
    2 = UP

    Адаптивный порог:
    threshold_t = clip(max(base_threshold, atr_mult * atr_pct_14_t),
                       min_threshold, max_threshold)
    """
    x = df.copy()

    x["future_return"] = x["close"].shift(-horizon) / x["close"] - 1.0

    raw_threshold = np.maximum(base_threshold, atr_mult * x[atr_col])
    x["target_threshold"] = raw_threshold.clip(lower=min_threshold, upper=max_threshold)

    x["target"] = 1
    x.loc[x["future_return"] > x["target_threshold"], "target"] = 2
    x.loc[x["future_return"] < -x["target_threshold"], "target"] = 0

    return x


## Собрать датасет

In [16]:
def make_dataset(
    df: pd.DataFrame,
    horizon: int = 12,
    base_threshold: float = 0.0035,
    atr_mult: float = 0.7,
    min_threshold: float = 0.0030,
    max_threshold: float = 0.0080,
) -> pd.DataFrame:
    """
    Собирает фичи + target и удаляет строки с NaN.
    Leakage на границе train/test убирается не здесь,
    а в time_split / generate_walk_forward_splits через purge.
    """
    x = build_features(df)
    x = build_target(
        x,
        horizon=horizon,
        base_threshold=base_threshold,
        atr_mult=atr_mult,
        min_threshold=min_threshold,
        max_threshold=max_threshold,
    )
    x = x.dropna().copy()
    return x

## Выделить признаки

In [17]:
FEATURE_COLUMNS = [
    "ret_1", "ret_2", "ret_3", "ret_5", "ret_10", "ret_20",
    "hl_range", "body", "upper_wick", "lower_wick", "close_pos_in_range",
    "dist_ema_9", "dist_ema_21", "dist_ema_50",
    "ema_spread_9_21", "ema_spread_21_50",
    "ema_slope_9_3", "ema_slope_21_5", "ema_slope_50_10",
    "rsi_14", "rsi_delta_3",
    "atr_pct_14",
    "volatility_10", "volatility_20", "volatility_ratio_10_20",
    "volume_ratio", "volume_zscore_20",
    "dist_high_20", "dist_low_20",
    "range_expansion_1", "range_expansion_5",
    "breakout_up_20", "breakout_down_20",
    "trend_strength_20",
    "efficiency_ratio_10",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]


## Разделение train/test

In [18]:
def time_split(
    df: pd.DataFrame,
    train_size: float = 0.8,
    horizon: int = 12,
    embargo: int = 0
):
    """
    Делит датасет по времени: старое в train, новое в test.
    Вырезает последние horizon строк из train, чтобы их target
    не смотрел в будущее test-отрезка.
    """
    split_idx = int(len(df) * train_size)

    train_end = split_idx - horizon - embargo
    test_start = split_idx + embargo

    if train_end <= 0:
        raise ValueError("Слишком большой horizon/embargo для такого train_size")

    train_df = df.iloc[:train_end].copy()
    test_df = df.iloc[test_start:].copy()

    return train_df, test_df


## Обучение

In [19]:
def _make_prefit_calibrator(prefit_model, method: str = "sigmoid"):
    if FrozenEstimator is not None:
        return CalibratedClassifierCV(
            estimator=FrozenEstimator(prefit_model),
            method=method
        )

    # fallback для старых версий sklearn
    return CalibratedClassifierCV(
        estimator=prefit_model,
        method=method,
        cv="prefit"
    )


def train_xgboost_model(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    calibration_frac: float = 0.20,
    calibration_method: str = "sigmoid",
    use_calibration: bool = False
):
    split_idx = int(len(train_df) * (1 - calibration_frac))
    if split_idx <= 0 or split_idx >= len(train_df):
        raise ValueError("Некорректный calibration_frac")

    train_core = train_df.iloc[:split_idx].copy()
    calib_df = train_df.iloc[split_idx:].copy()

    X_train = train_core[FEATURE_COLUMNS]
    y_train = train_core["target"]

    X_test = test_df[FEATURE_COLUMNS]
    y_test = test_df["target"]

    classes = np.array([0, 1, 2])
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = dict(zip(classes, weights))
    sample_weights = y_train.map(class_weights)

    base_model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=400,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.5,
        random_state=42,
        eval_metric="mlogloss"
    )

    base_model.fit(X_train, y_train, sample_weight=sample_weights)

    model = base_model

    if use_calibration:
        X_calib = calib_df[FEATURE_COLUMNS]
        y_calib = calib_df["target"]

        calibrated_model = _make_prefit_calibrator(
            prefit_model=base_model,
            method=calibration_method
        )
        calibrated_model.fit(X_calib, y_calib)
        model = calibrated_model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    print("Class weights:", class_weights)
    print(f"Train core: {len(train_core)}, calibration: {len(calib_df)}, test: {len(test_df)}")
    print(f"Calibration enabled: {use_calibration}")
    print()

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()

    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    predictions_df = pd.DataFrame({
        "y_true": y_test.values,
        "y_pred": y_pred
    }, index=test_df.index)

    return model, predictions_df, y_prob


## Получить вероятность

In [20]:
def predict_probabilities(model, df: pd.DataFrame) -> pd.DataFrame:
    X = df[FEATURE_COLUMNS]
    probs = model.predict_proba(X)

    result = pd.DataFrame(
        probs,
        index=df.index,
        columns=["prob_down", "prob_unsure", "prob_up"]
    )

    result["pred_class_id"] = probs.argmax(axis=1)

    class_map = {
        0: "DOWN",
        1: "UNSURE",
        2: "UP",
    }
    result["pred_class"] = result["pred_class_id"].map(class_map)

    return result

In [21]:
def apply_decision_layer(
    preds: pd.DataFrame,
    min_direction_prob: float = 0.34,
    min_direction_edge: float = 0.03,
    max_unsure_prob: float = 0.50,
    use_unsure_cap: bool = True
) -> pd.DataFrame:
    x = preds.copy()

    x["direction_prob"] = x[["prob_down", "prob_up"]].max(axis=1)
    x["direction_edge"] = (x["prob_up"] - x["prob_down"]).abs()
    x["dominant_direction"] = np.where(
        x["prob_up"] >= x["prob_down"],
        "UP",
        "DOWN"
    )

    condition = (
        (x["direction_prob"] >= min_direction_prob) &
        (x["direction_edge"] >= min_direction_edge)
    )

    if use_unsure_cap:
        condition &= (x["prob_unsure"] <= max_unsure_prob)

    x["decision_class"] = np.where(
        condition,
        x["dominant_direction"],
        "UNSURE"
    )

    x["bias_score"] = x["prob_up"] - x["prob_down"]
    x["confidence_score"] = x[["prob_up", "prob_down"]].max(axis=1) - x["prob_unsure"]

    x["bias_label"] = np.select(
        [
            x["bias_score"] > min_direction_edge,
            x["bias_score"] < -min_direction_edge,
        ],
        [
            "BULLISH_BIAS",
            "BEARISH_BIAS",
        ],
        default="NEUTRAL_BIAS"
    )

    return x


In [24]:
preds = predict_probabilities(model, test_df)
preds = apply_decision_layer(preds)

print(preds["decision_class"].value_counts(normalize=True))

decision_class
DOWN      0.390336
UP        0.326490
UNSURE    0.283175
Name: proportion, dtype: float64


## Работа

In [26]:
# 1. Загружаем данные
df = load_binance_klines("BTCUSDT", "1h", 20_000)

# 2. Валидируем
df = validate_ohlcv(df, expected_freq="1h")

# 3. Собираем dataset
HORIZON = 12
dataset = make_dataset(
    df,
    horizon=HORIZON,
    base_threshold=0.0035,
    atr_mult=0.7,
    min_threshold=0.0030,
    max_threshold=0.0080,
)

# 4. Смотрим баланс классов
print("\nРаспределение target:")
print(dataset["target"].value_counts())
print()
print(dataset["target"].value_counts(normalize=True).sort_index())

# 5. Делим по времени с purge
train_df, test_df = time_split(
    dataset,
    train_size=0.8,
    horizon=HORIZON,
    embargo=0
)

print(f"\nTrain size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

# 6. Обучаем baseline, calibration по умолчанию выключен
model, predictions_df, y_prob = train_xgboost_model(
    train_df,
    test_df,
    calibration_frac=0.20,
    calibration_method="sigmoid",
    use_calibration=False
)

# 7. Получаем вероятности по test
preds = predict_probabilities(model, test_df)
preds = apply_decision_layer(
    preds,
    min_direction_prob=0.34,
    min_direction_edge=0.03,
    max_unsure_prob=0.50,
    use_unsure_cap=True
)

print(preds["decision_class"].value_counts(normalize=True))
print()
print(preds.tail())

result_with_price = test_df[[
    "close", "future_return", "target", "target_threshold"
]].join(preds)

print(result_with_price.tail(20))
print()
print(result_with_price[[
    "prob_down", "prob_unsure", "prob_up",
    "bias_score", "confidence_score"
]].describe())


Количество строк: 20000
Начало: 2024-01-05 00:00:00+00:00
Конец: 2026-04-17 07:00:00+00:00
Пропусков нет

Распределение target:
target
2    7148
0    6563
1    6255
Name: count, dtype: int64

target
0    0.328709
1    0.313283
2    0.358009
Name: proportion, dtype: float64

Train size: 15960
Test size: 3994
Class weights: {np.int64(0): np.float64(1.0295113691340105), np.int64(1): np.float64(1.0843312101910827), np.int64(2): np.float64(0.9038012316840094)}
Train core: 12768, calibration: 3192, test: 3994
Calibration enabled: False

Confusion Matrix:
[[618 378 464]
 [366 527 343]
 [533 303 462]]

Classification Report:
              precision    recall  f1-score   support

           0     0.4074    0.4233    0.4152      1460
           1     0.4363    0.4264    0.4313      1236
           2     0.3641    0.3559    0.3600      1298

    accuracy                         0.4024      3994
   macro avg     0.4026    0.4019    0.4021      3994
weighted avg     0.4022    0.4024    0.4022      

In [27]:
import numpy as np
import pandas as pd

# =========================
# 1. Подготовка данных
# =========================

analysis_df = preds.copy()

# Если в preds вдруг нет future_return / target, попробуем подтянуть из test_df
needed_cols = ["future_return", "target"]
missing_cols = [c for c in needed_cols if c not in analysis_df.columns]

if missing_cols:
    if "test_df" not in globals():
        raise ValueError(
            f"В preds не хватает колонок {missing_cols}, и test_df не найден."
        )
    join_cols = [c for c in needed_cols if c in test_df.columns]
    analysis_df = analysis_df.join(test_df[join_cols], how="left")

if "future_return" not in analysis_df.columns:
    raise ValueError("Не найдена колонка future_return.")

# На всякий случай убираем пропуски в ключевых полях
analysis_df = analysis_df.dropna(subset=["bias_score", "confidence_score", "future_return"]).copy()

# =========================
# 2. Бины по bias_score
# =========================

bias_bins = [-np.inf, -0.20, -0.10, -0.03, 0.03, 0.10, 0.20, np.inf]
bias_labels = [
    "< -0.20",
    "-0.20 .. -0.10",
    "-0.10 .. -0.03",
    "-0.03 .. 0.03",
    "0.03 .. 0.10",
    "0.10 .. 0.20",
    "> 0.20"
]

analysis_df["bias_bin"] = pd.cut(
    analysis_df["bias_score"],
    bins=bias_bins,
    labels=bias_labels,
    include_lowest=True,
    ordered=True
)

# Доп. признаки для аналитики
analysis_df["is_up_move"] = (analysis_df["future_return"] > 0).astype(int)
analysis_df["is_down_move"] = (analysis_df["future_return"] < 0).astype(int)
analysis_df["abs_future_return"] = analysis_df["future_return"].abs()

# =========================
# 3. Итоговая таблица по бинам
# =========================

bin_table = (
    analysis_df
    .groupby("bias_bin", observed=False)
    .agg(
        n=("future_return", "size"),
        avg_future_return=("future_return", "mean"),
        median_future_return=("future_return", "median"),
        std_future_return=("future_return", "std"),
        up_rate=("is_up_move", "mean"),
        down_rate=("is_down_move", "mean"),
        avg_abs_future_return=("abs_future_return", "mean"),
        avg_confidence=("confidence_score", "mean"),
        avg_prob_up=("prob_up", "mean"),
        avg_prob_down=("prob_down", "mean"),
        avg_prob_unsure=("prob_unsure", "mean"),
    )
    .reset_index()
)

# Добавим coverage
bin_table["coverage"] = bin_table["n"] / len(analysis_df)

# Для удобства округлим
numeric_cols = bin_table.select_dtypes(include=[np.number]).columns
bin_table[numeric_cols] = bin_table[numeric_cols].round(4)

print("=== Таблица по бинам bias_score ===")
print(bin_table)

# =========================
# 4. Качество decision_class отдельно по UP / DOWN
# =========================

directional_df = analysis_df[analysis_df["decision_class"].isin(["UP", "DOWN"])].copy()

if len(directional_df) > 0:
    directional_df["direction_hit"] = np.where(
        ((directional_df["decision_class"] == "UP") & (directional_df["future_return"] > 0)) |
        ((directional_df["decision_class"] == "DOWN") & (directional_df["future_return"] < 0)),
        1,
        0
    )

    direction_table = (
        directional_df
        .groupby("decision_class", observed=False)
        .agg(
            n=("future_return", "size"),
            hit_rate=("direction_hit", "mean"),
            avg_future_return=("future_return", "mean"),
            median_future_return=("future_return", "median"),
            avg_abs_future_return=("abs_future_return", "mean"),
            avg_bias_score=("bias_score", "mean"),
            avg_confidence=("confidence_score", "mean"),
        )
        .reset_index()
    )

    numeric_cols = direction_table.select_dtypes(include=[np.number]).columns
    direction_table[numeric_cols] = direction_table[numeric_cols].round(4)

    print("\n=== Качество decision_class по направлениям ===")
    print(direction_table)
else:
    print("\nНет строк с decision_class == UP/DOWN для directional-анализа.")

# =========================
# 5. Простая связь bias_score и future_return
# =========================

pearson_corr = analysis_df["bias_score"].corr(analysis_df["future_return"], method="pearson")
spearman_corr = analysis_df["bias_score"].corr(analysis_df["future_return"], method="spearman")

print("\n=== Корреляция bias_score vs future_return ===")
print(f"Pearson:  {pearson_corr:.4f}")
print(f"Spearman: {spearman_corr:.4f}")

# =========================
# 6. Быстрый вывод по крайним зонам
# =========================

extreme_negative = analysis_df[analysis_df["bias_score"] <= -0.20]
extreme_positive = analysis_df[analysis_df["bias_score"] >= 0.20]

print("\n=== Крайние зоны bias_score ===")

if len(extreme_negative) > 0:
    print(f"Strong bearish (bias_score <= -0.20): n={len(extreme_negative)}")
    print(f"  avg_future_return = {extreme_negative['future_return'].mean():.4f}")
    print(f"  up_rate           = {extreme_negative['is_up_move'].mean():.4f}")
    print(f"  down_rate         = {extreme_negative['is_down_move'].mean():.4f}")
else:
    print("Strong bearish: нет наблюдений")

if len(extreme_positive) > 0:
    print(f"Strong bullish (bias_score >= 0.20): n={len(extreme_positive)}")
    print(f"  avg_future_return = {extreme_positive['future_return'].mean():.4f}")
    print(f"  up_rate           = {extreme_positive['is_up_move'].mean():.4f}")
    print(f"  down_rate         = {extreme_positive['is_down_move'].mean():.4f}")
else:
    print("Strong bullish: нет наблюдений")

=== Таблица по бинам bias_score ===
         bias_bin    n  avg_future_return  median_future_return  \
0         < -0.20  635            -0.0028               -0.0019   
1  -0.20 .. -0.10  600            -0.0009               -0.0007   
2  -0.10 .. -0.03  598            -0.0018               -0.0013   
3   -0.03 .. 0.03  520             0.0007               -0.0000   
4    0.03 .. 0.10  548            -0.0012               -0.0003   
5    0.10 .. 0.20  553             0.0002               -0.0004   
6          > 0.20  540            -0.0007                0.0003   

   std_future_return  up_rate  down_rate  avg_abs_future_return  \
0             0.0171   0.4346     0.5654                 0.0127   
1             0.0172   0.4917     0.5083                 0.0127   
2             0.0181   0.4682     0.5318                 0.0130   
3             0.0172   0.5000     0.5000                 0.0125   
4             0.0154   0.4854     0.5146                 0.0109   
5             0.0179   0.

# WFA

## Фолды

In [ ]:
def generate_walk_forward_splits(
    df: pd.DataFrame,
    train_size: int,
    test_size: int,
    step_size: int,
    horizon: int = 12,
    embargo: int = 0
):
    """
    Генерирует последовательные train/test splits по времени.
    Expanding window + purge между train и test:
    последние horizon строк train вырезаются, чтобы не было leakage по target.
    """
    splits = []
    n = len(df)

    split_anchor = train_size

    while True:
        train_end = split_anchor - horizon - embargo
        test_start = split_anchor + embargo
        test_end = test_start + test_size

        if train_end <= 0 or test_end > n:
            break

        train_df = df.iloc[:train_end].copy()
        test_df = df.iloc[test_start:test_end].copy()

        splits.append((train_df, test_df))
        split_anchor += step_size

    return splits


In [ ]:
splits = generate_walk_forward_splits(
    dataset,
    train_size=10000,
    test_size=2000,
    step_size=2000,
    horizon=HORIZON,
    embargo=0
)

print(f"Количество фолдов: {len(splits)}")
for i, (train_df, test_df) in enumerate(splits, 1):
    print(i, len(train_df), len(test_df), train_df.index.min(), test_df.index.max())


## Метрики по raw multiclass prediction

In [ ]:
def calculate_classification_metrics(y_true, y_pred) -> dict:
    metrics = {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "recall_down": recall_score(y_true, y_pred, labels=[0], average="macro", zero_division=0),
        "recall_unsure": recall_score(y_true, y_pred, labels=[1], average="macro", zero_division=0),
        "recall_up": recall_score(y_true, y_pred, labels=[2], average="macro", zero_division=0),
    }
    return metrics

## Метрика по decision layer

In [ ]:
TARGET_TO_LABEL = {
    0: "DOWN",
    1: "UNSURE",
    2: "UP",
}
def calculate_directional_precision(
    y_true_numeric: pd.Series,
    decision_class: pd.Series
) -> float:
    """
    Считаем precision только на тех строках, где решение UP/DOWN.
    UNSURE исключаем.
    """
    y_true_label = y_true_numeric.map(TARGET_TO_LABEL)

    mask = decision_class.isin(["UP", "DOWN"])
    if mask.sum() == 0:
        return 0.0

    return (y_true_label[mask] == decision_class[mask]).mean()
    
def calculate_unsure_rate(decision_class: pd.Series) -> float:
    return (decision_class == "UNSURE").mean()

## Decision layer

In [ ]:
def apply_decision_layer(
    preds: pd.DataFrame,
    min_direction_prob: float = 0.34,
    min_direction_edge: float = 0.03,
    max_unsure_prob: float = 0.50,
    use_unsure_cap: bool = True
) -> pd.DataFrame:
    x = preds.copy()

    x["direction_prob"] = x[["prob_down", "prob_up"]].max(axis=1)
    x["direction_edge"] = (x["prob_up"] - x["prob_down"]).abs()
    x["dominant_direction"] = np.where(
        x["prob_up"] >= x["prob_down"],
        "UP",
        "DOWN"
    )

    condition = (
        (x["direction_prob"] >= min_direction_prob) &
        (x["direction_edge"] >= min_direction_edge)
    )

    if use_unsure_cap:
        condition &= (x["prob_unsure"] <= max_unsure_prob)

    x["decision_class"] = np.where(
        condition,
        x["dominant_direction"],
        "UNSURE"
    )

    x["bias_score"] = x["prob_up"] - x["prob_down"]
    x["confidence_score"] = x[["prob_up", "prob_down"]].max(axis=1) - x["prob_unsure"]

    x["bias_label"] = np.select(
        [
            x["bias_score"] > min_direction_edge,
            x["bias_score"] < -min_direction_edge,
        ],
        [
            "BULLISH_BIAS",
            "BEARISH_BIAS",
        ],
        default="NEUTRAL_BIAS"
    )

    return x


## Функция оценки фолда

In [ ]:
def evaluate_one_fold(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    decision_params: dict | None = None,
    calibration_frac: float = 0.20,
    calibration_method: str = "sigmoid",
    use_calibration: bool = False
) -> dict:
    if decision_params is None:
        decision_params = {
            "min_direction_prob": 0.34,
            "min_direction_edge": 0.03,
            "max_unsure_prob": 0.50,
            "use_unsure_cap": True,
        }

    model, predictions_df, y_prob = train_xgboost_model(
        train_df,
        test_df,
        calibration_frac=calibration_frac,
        calibration_method=calibration_method,
        use_calibration=use_calibration
    )

    preds = predict_probabilities(model, test_df)
    preds = apply_decision_layer(preds, **decision_params)

    y_true = test_df["target"]
    y_pred = preds["pred_class_id"]

    clf_metrics = calculate_classification_metrics(y_true, y_pred)
    directional_precision = calculate_directional_precision(y_true, preds["decision_class"])
    unsure_rate = calculate_unsure_rate(preds["decision_class"])
    mean_abs_bias = preds["bias_score"].abs().mean()
    mean_confidence_score = preds["confidence_score"].mean()

    result = {
        **clf_metrics,
        "directional_precision": directional_precision,
        "unsure_rate": unsure_rate,
        "mean_abs_bias": mean_abs_bias,
        "mean_confidence_score": mean_confidence_score,
        "n_test": len(test_df),
    }
    return result


In [ ]:
def run_walk_forward_evaluation(
    dataset: pd.DataFrame,
    train_size: int,
    test_size: int,
    step_size: int,
    horizon: int = 12,
    embargo: int = 0,
    decision_params: dict | None = None,
    calibration_frac: float = 0.20,
    calibration_method: str = "sigmoid",
    use_calibration: bool = False
) -> pd.DataFrame:
    splits = generate_walk_forward_splits(
        df=dataset,
        train_size=train_size,
        test_size=test_size,
        step_size=step_size,
        horizon=horizon,
        embargo=embargo
    )

    results = []

    for fold_idx, (train_df, test_df) in enumerate(splits, start=1):
        print(f"Fold {fold_idx}: train={len(train_df)}, test={len(test_df)}")

        metrics = evaluate_one_fold(
            train_df=train_df,
            test_df=test_df,
            decision_params=decision_params,
            calibration_frac=calibration_frac,
            calibration_method=calibration_method,
            use_calibration=use_calibration
        )
        metrics["fold"] = fold_idx
        metrics["train_end"] = train_df.index.max()
        metrics["test_end"] = test_df.index.max()

        results.append(metrics)

    return pd.DataFrame(results)


In [ ]:
wf_results = run_walk_forward_evaluation(
    dataset=dataset,
    train_size=10000,
    test_size=2000,
    step_size=2000,
    horizon=HORIZON,
    embargo=0,
    decision_params={
        "min_direction_prob": 0.34,
        "min_direction_edge": 0.03,
        "max_unsure_prob": 0.50,
        "use_unsure_cap": True,
    },
    calibration_frac=0.20,
    calibration_method="sigmoid",
    use_calibration=False
)

print(wf_results)
print()
print("Средние метрики:")
print(wf_results.mean(numeric_only=True))


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from itertools import product
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier


# =========================================================
# 1. Локальный purged walk-forward split
# =========================================================
def make_local_walk_forward_splits(
    dataset: pd.DataFrame,
    purge: int,
    min_train_size: int = 8000,
    test_size: int = 2000,
    step_size: int | None = None,
    max_folds: int = 4,
):
    x = dataset.sort_index().copy()

    if step_size is None:
        step_size = test_size

    folds = []
    train_end = min_train_size

    while train_end + purge + test_size <= len(x):
        test_start = train_end + purge
        test_end = test_start + test_size

        train_df = x.iloc[:train_end].copy()
        test_df = x.iloc[test_start:test_end].copy()

        folds.append((train_df, test_df))
        train_end += step_size

    if max_folds is not None and len(folds) > max_folds:
        folds = folds[-max_folds:]

    return folds


# =========================================================
# 2. Аккуратная калибровка prefit-модели
# =========================================================
def calibrate_prefit_model(base_model, X_calib, y_calib, method="sigmoid"):
    try:
        calibrator = CalibratedClassifierCV(
            estimator=base_model,
            method=method,
            cv="prefit"
        )
    except TypeError:
        calibrator = CalibratedClassifierCV(
            base_estimator=base_model,
            method=method,
            cv="prefit"
        )

    calibrator.fit(X_calib, y_calib)
    return calibrator


# =========================================================
# 3. Тренировка модели для поиска
# =========================================================
def train_model_for_search(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_columns: list[str],
    use_calibration: bool = False,
    calibration_frac: float = 0.20,
    calibration_method: str = "sigmoid",
    model_params: dict | None = None,
):
    if model_params is None:
        model_params = {
            "objective": "multi:softprob",
            "num_class": 3,
            "n_estimators": 400,
            "max_depth": 5,
            "learning_rate": 0.04,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_lambda": 1.5,
            "random_state": 42,
            "eval_metric": "mlogloss",
        }

    train_work = train_df.copy()

    # train / calibration split только если калибровка включена
    if use_calibration:
        split_idx = int(len(train_work) * (1 - calibration_frac))
        if split_idx <= 0 or split_idx >= len(train_work):
            raise ValueError("Некорректный calibration_frac")

        train_core = train_work.iloc[:split_idx].copy()
        calib_df = train_work.iloc[split_idx:].copy()
    else:
        train_core = train_work.copy()
        calib_df = None

    X_train = train_core[feature_columns]
    y_train = train_core["target"]

    X_test = test_df[feature_columns]
    y_test = test_df["target"]

    all_classes = np.array([0, 1, 2])
    present_classes = np.sort(y_train.unique())

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_train
    )

    class_weights = {0: 1.0, 1: 1.0, 2: 1.0}
    for c, w in zip(present_classes, weights):
        class_weights[int(c)] = float(w)

    sample_weights = y_train.map(class_weights).astype(float)

    base_model = XGBClassifier(**model_params)
    base_model.fit(X_train, y_train, sample_weight=sample_weights)

    model = base_model

    if use_calibration:
        X_calib = calib_df[feature_columns]
        y_calib = calib_df["target"]

        # Калибровать имеет смысл только если есть все классы
        if len(np.unique(y_calib)) == 3:
            model = calibrate_prefit_model(
                base_model,
                X_calib,
                y_calib,
                method=calibration_method
            )

    y_prob = model.predict_proba(X_test)
    return model, y_prob


# =========================================================
# 4. Построение prediction frame
# =========================================================
def build_prediction_frame(test_df: pd.DataFrame, y_prob: np.ndarray) -> pd.DataFrame:
    preds = pd.DataFrame(
        y_prob,
        index=test_df.index,
        columns=["prob_down", "prob_unsure", "prob_up"]
    )

    preds["pred_class_id"] = np.argmax(
        preds[["prob_down", "prob_unsure", "prob_up"]].values,
        axis=1
    )

    class_map = {0: "DOWN", 1: "UNSURE", 2: "UP"}
    preds["pred_class"] = preds["pred_class_id"].map(class_map)

    preds = preds.join(
        test_df[["future_return", "target"]],
        how="left"
    )

    return preds


# =========================================================
# 5. Асимметричный decision layer
# =========================================================
def apply_asymmetric_decision_layer(
    preds: pd.DataFrame,
    down_min_prob: float = 0.34,
    down_min_edge: float = 0.03,
    down_max_unsure: float = 0.55,
    up_min_prob: float = 0.40,
    up_min_edge: float = 0.06,
    up_max_unsure: float = 0.45,
    use_unsure_cap: bool = True,
) -> pd.DataFrame:
    x = preds.copy()

    x["direction_prob"] = x[["prob_down", "prob_up"]].max(axis=1)
    x["direction_edge"] = (x["prob_up"] - x["prob_down"]).abs()
    x["dominant_direction"] = np.where(
        x["prob_up"] >= x["prob_down"],
        "UP",
        "DOWN"
    )

    bearish_ok = (
        (x["prob_down"] >= down_min_prob) &
        ((x["prob_down"] - x["prob_up"]) >= down_min_edge)
    )

    bullish_ok = (
        (x["prob_up"] >= up_min_prob) &
        ((x["prob_up"] - x["prob_down"]) >= up_min_edge)
    )

    if use_unsure_cap:
        bearish_ok &= (x["prob_unsure"] <= down_max_unsure)
        bullish_ok &= (x["prob_unsure"] <= up_max_unsure)

    x["decision_class"] = "UNSURE"
    x.loc[bearish_ok, "decision_class"] = "DOWN"
    x.loc[bullish_ok, "decision_class"] = "UP"

    x["bias_score"] = x["prob_up"] - x["prob_down"]
    x["confidence_score"] = x[["prob_up", "prob_down"]].max(axis=1) - x["prob_unsure"]

    x["bias_label"] = np.select(
        [
            x["bias_score"] > 0.03,
            x["bias_score"] < -0.03,
        ],
        [
            "BULLISH_BIAS",
            "BEARISH_BIAS",
        ],
        default="NEUTRAL_BIAS"
    )

    return x


# =========================================================
# 6. Метрики по одному prediction frame
# =========================================================
def summarize_prediction_frame(preds: pd.DataFrame) -> dict:
    x = preds.copy()

    n_rows = len(x)
    directional = x[x["decision_class"].isin(["DOWN", "UP"])].copy()
    down_df = x[x["decision_class"] == "DOWN"].copy()
    up_df = x[x["decision_class"] == "UP"].copy()

    x["is_up_move"] = (x["future_return"] > 0).astype(int)
    x["is_down_move"] = (x["future_return"] < 0).astype(int)

    directional_hit_rate = np.nan
    if len(directional) > 0:
        directional_hit_rate = np.mean(
            ((directional["decision_class"] == "UP") & (directional["future_return"] > 0)) |
            ((directional["decision_class"] == "DOWN") & (directional["future_return"] < 0))
        )

    down_hit_rate = np.nan
    if len(down_df) > 0:
        down_hit_rate = (down_df["future_return"] < 0).mean()

    up_hit_rate = np.nan
    if len(up_df) > 0:
        up_hit_rate = (up_df["future_return"] > 0).mean()

    bias_pearson = x["bias_score"].corr(x["future_return"], method="pearson")
    bias_spearman = x["bias_score"].corr(x["future_return"], method="spearman")

    out = {
        "n_rows": n_rows,
        "coverage": len(directional) / n_rows if n_rows else np.nan,
        "unsure_rate": (x["decision_class"] == "UNSURE").mean(),
        "down_share": len(down_df) / n_rows if n_rows else np.nan,
        "up_share": len(up_df) / n_rows if n_rows else np.nan,
        "down_n": len(down_df),
        "up_n": len(up_df),
        "directional_hit_rate": directional_hit_rate,
        "down_hit_rate": down_hit_rate,
        "up_hit_rate": up_hit_rate,
        "down_avg_future_return": down_df["future_return"].mean() if len(down_df) else np.nan,
        "up_avg_future_return": up_df["future_return"].mean() if len(up_df) else np.nan,
        "down_avg_abs_future_return": down_df["future_return"].abs().mean() if len(down_df) else np.nan,
        "up_avg_abs_future_return": up_df["future_return"].abs().mean() if len(up_df) else np.nan,
        "bias_pearson": bias_pearson,
        "bias_spearman": bias_spearman,
        "avg_bias_score": x["bias_score"].mean(),
        "avg_confidence_score": x["confidence_score"].mean(),
    }

    return out


# =========================================================
# 7. Ранжирующий score для поиска
#    Это не "истина", а удобный критерий сортировки.
# =========================================================
def compute_search_score(row: dict) -> float:
    down_hit_edge = (row["down_hit_rate"] - 0.50) if pd.notna(row["down_hit_rate"]) else -0.03
    up_hit_edge = (row["up_hit_rate"] - 0.50) if pd.notna(row["up_hit_rate"]) else -0.03

    down_return_edge = (-row["down_avg_future_return"]) if pd.notna(row["down_avg_future_return"]) else -0.001
    up_return_edge = row["up_avg_future_return"] if pd.notna(row["up_avg_future_return"]) else -0.001

    bias_rank = row["bias_spearman"] if pd.notna(row["bias_spearman"]) else 0.0
    coverage = row["coverage"] if pd.notna(row["coverage"]) else 0.0

    penalty = 0.0

    # Наказание за слишком маленькое число directional-сигналов
    if row["down_share"] < 0.10:
        penalty += 0.04
    if row["up_share"] < 0.08:
        penalty += 0.04

    # Наказание за слишком маленький общий coverage
    if coverage < 0.25:
        penalty += (0.25 - coverage) * 0.40

    # Наказание, если bullish-часть явно плохая
    if pd.notna(row["up_hit_rate"]) and row["up_hit_rate"] < 0.48:
        penalty += 0.02

    score = (
        1.50 * down_hit_edge +
        0.75 * up_hit_edge +
        20.0 * down_return_edge +
        12.0 * up_return_edge +
        0.50 * bias_rank +
        0.15 * coverage -
        penalty
    )

    return float(score)


# =========================================================
# 8. Оценка одного конфига на walk-forward
# =========================================================
def evaluate_config_on_walk_forward(
    df_raw: pd.DataFrame,
    config: dict,
    feature_columns: list[str],
    min_train_size: int = 8000,
    test_size: int = 2000,
    step_size: int | None = None,
    max_folds: int = 4,
    calibration_frac: float = 0.20,
    calibration_method: str = "sigmoid",
    model_params: dict | None = None,
):
    dataset = make_dataset(
        df_raw,
        horizon=config["horizon"],
        base_threshold=config["base_threshold"],
        atr_mult=config["atr_mult"],
        min_threshold=config["min_threshold"],
        max_threshold=config["max_threshold"],
    )

    folds = make_local_walk_forward_splits(
        dataset=dataset,
        purge=config["horizon"],
        min_train_size=min_train_size,
        test_size=test_size,
        step_size=step_size,
        max_folds=max_folds,
    )

    if len(folds) == 0:
        raise ValueError("Не удалось собрать ни одного walk-forward фолда.")

    fold_metrics = []

    for fold_id, (train_df, test_df) in enumerate(folds, start=1):
        model, y_prob = train_model_for_search(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            use_calibration=config["use_calibration"],
            calibration_frac=calibration_frac,
            calibration_method=calibration_method,
            model_params=model_params,
        )

        preds = build_prediction_frame(test_df, y_prob)

        preds = apply_asymmetric_decision_layer(
            preds,
            down_min_prob=config["down_min_prob"],
            down_min_edge=config["down_min_edge"],
            down_max_unsure=config["down_max_unsure"],
            up_min_prob=config["up_min_prob"],
            up_min_edge=config["up_min_edge"],
            up_max_unsure=config["up_max_unsure"],
            use_unsure_cap=True,
        )

        metrics = summarize_prediction_frame(preds)
        metrics["fold_id"] = fold_id
        fold_metrics.append(metrics)

    fold_metrics_df = pd.DataFrame(fold_metrics)

    agg = fold_metrics_df.mean(numeric_only=True).to_dict()
    agg["n_folds"] = len(fold_metrics_df)
    agg["total_rows"] = int(fold_metrics_df["n_rows"].sum())

    for k, v in config.items():
        agg[k] = v

    agg["score"] = compute_search_score(agg)
    agg["down_edge_bps"] = -agg["down_avg_future_return"] * 10000 if pd.notna(agg["down_avg_future_return"]) else np.nan
    agg["up_edge_bps"] = agg["up_avg_future_return"] * 10000 if pd.notna(agg["up_avg_future_return"]) else np.nan

    return agg, fold_metrics_df


# =========================================================
# 9. Узкое search space
#    Сначала гоняем это. Потом, если увидим осмысленный edge,
#    можно расширять.
# =========================================================
search_space = {
    "horizon": [8, 12, 16],
    "base_threshold": [0.0030, 0.0035],
    "atr_mult": [0.6, 0.8],
    "min_threshold": [0.0025, 0.0030],
    "max_threshold": [0.0070, 0.0080],
    "use_calibration": [False],  # сначала без калибровки
    "down_min_prob": [0.34, 0.38],
    "down_min_edge": [0.03, 0.06],
    "down_max_unsure": [0.55],
    "up_min_prob": [0.40, 0.44],
    "up_min_edge": [0.06, 0.10],
    "up_max_unsure": [0.45],
}

param_grid = list(ParameterGrid(search_space))
print(f"Количество конфигов в поиске: {len(param_grid)}")


# =========================================================
# 10. Запуск поиска
# =========================================================
results = []
fold_details = {}

for i, cfg in enumerate(param_grid, start=1):
    agg, fold_df = evaluate_config_on_walk_forward(
        df_raw=df,
        config=cfg,
        feature_columns=FEATURE_COLUMNS,
        min_train_size=8000,
        test_size=2000,
        step_size=2000,
        max_folds=4,
        calibration_frac=0.20,
        calibration_method="sigmoid",
        model_params=None,
    )

    agg["search_id"] = i
    results.append(agg)
    fold_details[i] = fold_df.copy()

    if i % 10 == 0 or i == len(param_grid):
        print(f"Готово: {i}/{len(param_grid)}")


# =========================================================
# 11. Таблица результатов
# =========================================================
results_df = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)

view_cols = [
    "search_id",
    "score",
    "horizon",
    "base_threshold",
    "atr_mult",
    "min_threshold",
    "max_threshold",
    "use_calibration",
    "coverage",
    "unsure_rate",
    "down_share",
    "up_share",
    "directional_hit_rate",
    "down_hit_rate",
    "up_hit_rate",
    "down_edge_bps",
    "up_edge_bps",
    "bias_spearman",
    "down_min_prob",
    "down_min_edge",
    "up_min_prob",
    "up_min_edge",
]

print("\n=== TOP-10 конфигов ===")
print(results_df[view_cols].head(10).round(4))


# =========================================================
# 12. Лучший конфиг и его фолды
# =========================================================
best_row = results_df.iloc[0].copy()
best_search_id = int(best_row["search_id"])

print("\n=== Лучший конфиг ===")
print(best_row[[
    "search_id",
    "score",
    "horizon",
    "base_threshold",
    "atr_mult",
    "min_threshold",
    "max_threshold",
    "use_calibration",
    "down_min_prob",
    "down_min_edge",
    "down_max_unsure",
    "up_min_prob",
    "up_min_edge",
    "up_max_unsure",
    "coverage",
    "unsure_rate",
    "down_hit_rate",
    "up_hit_rate",
    "down_edge_bps",
    "up_edge_bps",
    "bias_spearman",
]].round(4))

print("\n=== Детали лучшего конфига по фолдам ===")
print(fold_details[best_search_id].round(4))


# =========================================================
# 13. Сохраним результаты
# =========================================================
results_df.to_csv("wf_optimizer_results.csv", index=False)
print("\nФайл сохранён: wf_optimizer_results.csv")

Количество конфигов в поиске: 768
Готово: 10/768
Готово: 20/768
Готово: 30/768
Готово: 40/768
Готово: 50/768
Готово: 60/768
Готово: 70/768
Готово: 80/768
Готово: 90/768
Готово: 100/768
Готово: 110/768
Готово: 120/768
Готово: 130/768
Готово: 140/768
Готово: 150/768
Готово: 160/768
Готово: 170/768
Готово: 180/768
Готово: 190/768
Готово: 200/768
